# Least Squares and the Normal Equations

07/01 regressed one asset on one market return. Most finance regressions have several explanatory returns: a balanced fund
on equity and bond indices, a hedge on several futures, a stock on several factors. In matrix form all of them are one
formula, $\hat b = (X^\top X)^{-1}X^\top y$. This notebook derives it, reads it as a projection, gives the standard errors, and shows the two
things that go wrong in practice: correlated regressors and numerical ill-conditioning.

## 0. Math link

- [Least squares](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/0510_least%20squares.ipynb):
  the least-squares solution of an overdetermined system $Xb \approx y$ via the pseudoinverse.
- [Gradients](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/03_gradients.ipynb):
  setting the gradient of the squared error to zero gives the normal equations.
- [Covariance matrix](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/08_covariance_matrix.ipynb):
  the covariance matrix of the estimates is $\sigma^2(X^\top X)^{-1}$.
- The one-regressor case from [07/01](./01_beta_as_a_regression_slope.ipynb).

## 1. Introduction

A balanced fund claims to be "actively managed with a flexible allocation". Regressing its returns on Swiss equities, global
equities and Swiss bonds at once reveals what it actually holds and what it charges for. The same regression can mislead
when two regressors move almost together, and the textbook formula can lose digits on a computer.

After this notebook you can:

- write a multiple regression in matrix form and derive the normal equations;
- interpret the fitted values as an orthogonal projection and check the residual properties;
- compute standard errors and t-statistics from $\sigma^2(X^\top X)^{-1}$;
- measure multicollinearity with the variance inflation factor and see its effect on the estimates;
- solve least-squares problems stably (QR, `lstsq`) instead of inverting $X^\top X$;
- run a returns-based style analysis as a constrained least-squares problem.

## 2. Theory

### Matrix form

With $n$ observations and $k$ regressors (including a column of ones for the intercept):

$$
y = Xb + \varepsilon, \qquad y \in \mathbb{R}^n,\ X \in \mathbb{R}^{n\times k},\ b \in \mathbb{R}^k
$$

### The normal equations

Minimize $S(b) = \lVert y - Xb\rVert^2 = (y - Xb)^\top(y - Xb)$. Its gradient is

$$
\nabla S(b) = -2X^\top(y - Xb)
$$

Setting it to zero gives the **normal equations** and, if $X$ has full column rank, the solution:

$$
X^\top X\,\hat b = X^\top y, \qquad \hat b = (X^\top X)^{-1}X^\top y
$$

$(X^\top X)^{-1}X^\top$ is the pseudoinverse $X^+$ of a full-column-rank $X$. With one regressor plus intercept this reduces to
$\hat\beta = \widehat{\operatorname{Cov}}(x,y)/\widehat{\operatorname{Var}}(x)$ from 07/01.

### Geometry: projection

The fitted values $\hat y = X\hat b = Hy$ with the **hat matrix** $H = X(X^\top X)^{-1}X^\top$. $H$ is symmetric and idempotent ($H^2 = H$): it
projects $y$ orthogonally onto the column space of $X$. Consequences:

- the residuals $e = y - \hat y = (I - H)y$ are orthogonal to every column of $X$: $X^\top e = 0$;
- with an intercept column, the residuals sum to zero;
- $\operatorname{tr}(H) = k$, the number of parameters;
- $\lVert y - \bar y\rVert^2 = \lVert\hat y - \bar y\rVert^2 + \lVert e\rVert^2$, so $R^2 = 1 - \lVert e\rVert^2/\lVert y - \bar y\rVert^2$. Adding a regressor never lowers $R^2$.

### Standard errors

If $\varepsilon$ has independent entries with variance $\sigma^2$, then $\hat b - b = (X^\top X)^{-1}X^\top\varepsilon$ and

$$
\operatorname{Cov}(\hat b) = \sigma^2(X^\top X)^{-1}, \qquad s^2 = \frac{e^\top e}{n - k}
$$

The standard error of $\hat b_j$ is $s\sqrt{[(X^\top X)^{-1}]_{jj}}$, and $t_j = \hat b_j/\operatorname{SE}(\hat b_j)$.

### Multicollinearity

If regressor $j$ is well explained by the other regressors, with $R_j^2$ from regressing it on them, then

$$
\operatorname{Var}(\hat b_j) = \frac{\sigma^2}{n\,\widehat{\operatorname{Var}}(x_j)}\cdot\underbrace{\frac{1}{1 - R_j^2}}_{\text{VIF}_j}
$$

The **variance inflation factor** grows without bound as $R_j^2 \to 1$. With two regressors of correlation $\rho$, $\text{VIF} = 1/(1 - \rho^2)$. The
estimates of correlated regressors are negatively correlated with each other: the data can identify their **sum** well,
but not how to split it.

### Numerics

The condition number of $X^\top X$ is the square of that of $X$: $\kappa(X^\top X) = \kappa(X)^2$. Forming $X^\top X$ and inverting it loses about
twice as many digits as necessary. Stable methods work on $X$ directly: QR decomposition ($X = QR$, then solve $R\hat b = Q^\top y$) or the
SVD (`np.linalg.lstsq`). For two standardized regressors with correlation $\rho$, $\kappa(X^\top X) = (1 + \rho)/(1 - \rho)$.

### Constrained least squares: style analysis

Sharpe (1992) explains a fund's returns by a portfolio of asset-class indices with weights $w \ge 0$, $\sum w = 1$:

$$
\min_w \lVert r_{\text{fund}} - Rw\rVert^2 \quad\text{s.t.}\ w \ge 0,\ \textstyle\sum_i w_i = 1
$$

The solution is the fund's **effective asset mix**; the residual is selection, and its mean is alpha after the style
benchmark.

## 3. Worked Examples

### Example 1: normal equations by hand

The data of 07/01, Example 4: $x = (2, -1, 3, 0, 1)$, $y = (3, -2, 4, 1, 1)$ (%), $X = [\mathbf{1}, x]$.

$$
X^\top X = \begin{pmatrix} 5 & 5 \\ 5 & 15 \end{pmatrix}, \quad X^\top y = \begin{pmatrix} 7 \\ 21 \end{pmatrix}, \quad
(X^\top X)^{-1} = \frac{1}{50}\begin{pmatrix} 15 & -5 \\ -5 & 5 \end{pmatrix}, \quad
\hat b = \frac{1}{50}\begin{pmatrix} 105 - 105 \\ -35 + 105 \end{pmatrix} = \begin{pmatrix} 0 \\ 1.4 \end{pmatrix}
$$

The same as the covariance formula.

### Example 2: residuals are orthogonal

$e = y - 1.4x = (0.2, -0.6, -0.2, 1.0, -0.4)$. $\sum e = 0$ and $\sum x e = 0.4 + 0.6 - 0.6 + 0 - 0.4 = 0$.

### Example 3: variance inflation

Swiss and global equities have correlation 0.8: $\text{VIF} = 1/0.36 = 2.78$, standard errors $\sqrt{2.78} = 1.67$ times larger than with
uncorrelated regressors. Two broad Swiss indices with correlation 0.98: $\text{VIF} = 1/0.0396 = 25.3$, standard errors 5 times larger.

### Example 4: condition numbers

Standardized regressors with $\rho = 0.8$: $\kappa(X^\top X) = 1.8/0.2 = 9$. With $\rho = 0.98$: $1.98/0.02 = 99$. With $\rho = 0.999999$: about $2\cdot 10^6$, and
$\kappa(X) \approx 1{,}400$.

## 4. Finance Application

### Where it matters

- **Fund analysis.** Regressing a fund on several indices shows its effective exposures and the alpha left after them;
  style analysis does the same with weights that look like a portfolio.
- **Hedging with several instruments.** The minimum-variance hedge with $k$ futures is the regression slope vector of the
  position on the futures returns.
- **Factor models.** 07/03 regresses returns on several factors; the betas are the slopes, the attribution follows from
  $X\hat b$.
- **Choosing regressors.** Two highly correlated indices (SMI and SPI, MSCI World and S&P 500) in the same regression give
  unstable individual betas. Keep one, or combine them.

### The usual misreadings

- "The fund has a negative beta to global equities, so it is short global equities." With correlated regressors, an
  individual coefficient can take the wrong sign by chance; look at the standard errors and the VIF, and at the sum of the
  correlated coefficients.
- "Adding a regressor raised $R^2$, so the model is better." $R^2$ never falls when a regressor is added. Use adjusted $R^2$ or
  out-of-sample fit.
- "`inv(X.T @ X) @ X.T @ y` is the way to compute it." It works for well-conditioned problems and fails quietly for
  ill-conditioned ones.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

rng = np.random.default_rng(42)
months_per_year = 12

assets = ["Swiss equities", "global equities", "Swiss bonds"]
mu = np.array([0.065, 0.07, 0.02])
vols = np.array([0.16, 0.17, 0.05])
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma = np.outer(vols, vols) * corr
rf = 0.005

### A balanced fund

Ten years of monthly excess returns. The fund holds 30% Swiss equities, 30% global equities and 40% Swiss bonds, adds
selection noise with 1.5% tracking error, and charges 1% a year in fees (alpha −1%).

In [ ]:
n = 120
w_fund = np.array([0.3, 0.3, 0.4])
alpha_fund, te_fund = -0.01, 0.015

R = (mu - rf) / 12 + rng.standard_normal((n, 3)) @ np.linalg.cholesky(Sigma / 12).T     # monthly excess returns
y = alpha_fund / 12 + R @ w_fund + te_fund / np.sqrt(12) * rng.standard_normal(n)
X = np.column_stack([np.ones(n), R])

b_normal = np.linalg.solve(X.T @ X, X.T @ y)                  # normal equations
b_inv = np.linalg.inv(X.T @ X) @ X.T @ y                       # explicit inverse (avoid)
b_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]                 # SVD
Q, Rq = np.linalg.qr(X)
b_qr = np.linalg.solve(Rq, Q.T @ y)                            # QR

e = y - X @ b_lstsq
k = X.shape[1]
s2 = e @ e / (n - k)
se = np.sqrt(s2 * np.diag(np.linalg.inv(X.T @ X)))
names = ["intercept (monthly)"] + assets
reg = pd.DataFrame({"true": np.r_[alpha_fund / 12, w_fund], "estimate": b_lstsq, "SE": se, "t": b_lstsq / se}, index=names)
print("max difference between methods:", np.abs(np.c_[b_normal, b_inv, b_qr] - b_lstsq[:, None]).max())
print(f"alpha: {b_lstsq[0] * 12:.2%} per year, SE {se[0] * 12:.2%};  residual vol {np.sqrt(s2 * 12):.2%};  "
      f"R^2 {1 - e @ e / ((y - y.mean()) @ (y - y.mean())):.3f}")
reg.round(4)

All four methods agree to machine precision here, because the problem is well conditioned. The regression recovers the
holdings to within about one to two points (standard errors 1.4–3.2 points), the fee as an alpha of −1.05% per year
($t = -2.1$), and $R^2 = 0.97$: the "flexible allocation" is a fixed 30/30/40 mix.

### Projection checks

In [ ]:
H = X @ np.linalg.solve(X.T @ X, X.T)
print(f"symmetric: {np.allclose(H, H.T)},  idempotent: {np.allclose(H @ H, H)},  trace: {np.trace(H):.6f}")
print(f"X'e = {np.round(X.T @ e, 12)}")
y_hat = H @ y
print(f"sum of squares: total {((y - y.mean())**2).sum():.6f} = "
      f"explained {((y_hat - y.mean())**2).sum():.6f} + residual {(e**2).sum():.6f}")

### Multicollinearity

Regress a portfolio of two equity indices on both indices, with correlation 0.8 and 0.98, over 2,000 histories of 10 years.
The true coefficients are 0.5 and 0.5.

In [ ]:
def collinear_slopes(rho, n_paths=2_000, n_obs=120, rng=rng):
    C = np.array([[1, rho], [rho, 1]]) * 0.16**2 / 12
    L = np.linalg.cholesky(C)
    out = np.empty((n_paths, 2))
    for p in range(n_paths):
        Z = rng.standard_normal((n_obs, 2)) @ L.T
        yy = Z @ np.array([0.5, 0.5]) + 0.05 / np.sqrt(12) * rng.standard_normal(n_obs)
        out[p] = np.linalg.lstsq(np.column_stack([np.ones(n_obs), Z]), yy, rcond=None)[0][1:]
    return out

slopes = {rho: collinear_slopes(rho) for rho in [0.0, 0.8, 0.98]}
pd.DataFrame({
    rho: {"SE slope 1": s[:, 0].std(), "SE of sum": s.sum(axis=1).std(), "corr of the two estimates": np.corrcoef(s.T)[0, 1],
          "VIF": 1 / (1 - rho**2), "SE ratio vs rho=0": s[:, 0].std() / slopes[0.0][:, 0].std()}
    for rho, s in slopes.items()
}).round(3)

At correlation 0.98 each slope has a standard error five times larger than with uncorrelated regressors, exactly
$\sqrt{\text{VIF}}$, and the two estimates have correlation −0.98. Yet the standard error of their **sum** is 0.03 in all three
cases: the regression knows the total equity exposure, just not how to split it.

### Numerical accuracy

Two regressors with correlation extremely close to 1 and known true coefficients. Compare the relative error of the explicit
normal equations with QR and `lstsq`.

In [ ]:
def accuracy(one_minus_rho, n_obs=500, rng=rng):
    rho = 1 - one_minus_rho
    z1 = rng.standard_normal(n_obs)
    z2 = rho * z1 + np.sqrt(1 - rho**2) * rng.standard_normal(n_obs)
    Xa = np.column_stack([np.ones(n_obs), z1, z2])
    b_true = np.array([0.1, 0.5, 0.5])
    ya = Xa @ b_true                                            # no noise: exact answer is b_true
    errs = {}
    errs["normal equations"] = np.linalg.solve(Xa.T @ Xa, Xa.T @ ya)
    Qa, Ra = np.linalg.qr(Xa)
    errs["QR"] = np.linalg.solve(Ra, Qa.T @ ya)
    errs["lstsq"] = np.linalg.lstsq(Xa, ya, rcond=None)[0]
    rel = {k: np.linalg.norm(v - b_true) / np.linalg.norm(b_true) for k, v in errs.items()}
    rel["cond(X)"] = np.linalg.cond(Xa)
    return rel

gaps = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]
acc = pd.DataFrame({g: accuracy(g) for g in gaps}).T
acc.index.name = "1 - rho"
acc.style.format("{:.1e}")

The normal equations lose accuracy in proportion to $\kappa(X)^2$: a relative error of $7\cdot 10^{-6}$ at $\kappa(X) = 1.4\cdot 10^5$, while QR
and `lstsq` stay below $10^{-11}$. For return regressions $\kappa(X)$ is usually modest, but nearly duplicate regressors (two versions of
the same index, a portfolio and its components) push it up quickly.

### Style analysis

Constrained least squares for the balanced fund: weights non-negative and summing to one.

In [ ]:
def style_analysis(y, R):
    k = R.shape[1]
    obj = lambda w: np.sum((y - R @ w) ** 2)
    res = minimize(obj, np.full(k, 1 / k), method="SLSQP", bounds=[(0, 1)] * k,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}])
    return res.x

w_style = style_analysis(y, R)
selection = y - R @ w_style
print("style weights:", dict(zip(assets, w_style.round(3))))
print(f"mean selection return (alpha vs style benchmark): {selection.mean() * 12:.2%} per year, "
      f"SE {selection.std(ddof=1) * np.sqrt(12) / np.sqrt(10):.2%}")

The style weights, 30.5/28.9/40.6, are close to the true 30/30/40, and the selection return of −1.03% per year is the fee.
Here the constraints are not binding, so style analysis and the unconstrained regression nearly agree; for funds with
short or leveraged positions they differ.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(3)
ax.bar(x_pos - 0.2, w_fund, width=0.4, label="true holdings", color="grey")
ax.bar(x_pos + 0.2, b_lstsq[1:], width=0.4, yerr=1.96 * se[1:], capsize=4, label="regression estimate ± 1.96 SE")
ax.plot(x_pos + 0.2, w_style, "k_", markersize=20, mew=2, label="style analysis")
ax.set_xticks(x_pos)
ax.set_xticklabels(assets)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Balanced fund: true weights vs estimated exposures (10 years monthly)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for rho, c in [(0.98, "tab:red"), (0.8, "tab:blue"), (0.0, "tab:green")]:
    s = slopes[rho][:800]
    ax.scatter(s[:, 0], s[:, 1], s=5, alpha=0.4, color=c, label=f"rho = {rho}")
ax.plot(0.5, 0.5, "k*", markersize=12, label="true")
ax.plot([-0.5, 1.5], [1.5, -0.5], color="grey", linestyle=":", linewidth=0.8, label="slope 1 + slope 2 = 1")
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_title("Slope estimates of two correlated regressors")
ax.set_xlabel("slope on index 1")
ax.set_ylabel("slope on index 2")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
rho_grid = np.linspace(0, 0.99, 200)
axes[0].plot(rho_grid, np.sqrt(1 / (1 - rho_grid**2)), color="tab:red")
axes[0].set_title("Standard-error inflation sqrt(VIF), two regressors")
axes[0].set_xlabel("correlation between regressors")
axes[0].set_ylabel("SE multiplier")

for col, mk in [("normal equations", "o-"), ("QR", "s-"), ("lstsq", "^-")]:
    axes[1].loglog(acc["cond(X)"], acc[col].clip(lower=1e-17), mk, label=col)
axes[1].set_title("Relative error of the solution vs condition number of X")
axes[1].set_xlabel("cond(X)")
axes[1].set_ylabel("relative error")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Solve the normal equations by hand for $x = (1, 2, 3, 4)$, $y = (1, 3, 2, 5)$ with an intercept. Check with `np.linalg.lstsq`.
2. Two regressors have correlation 0.9. By what factor are the standard errors of their slopes inflated?
3. A fund marketed as "absolute return, uncorrelated with markets" has style weights 55% global equities and 45% Swiss bonds,
   with $R^2 = 0.93$ over five years. **Interpret** this for an investor.

### Intermediate

4. Add a regressor of pure noise (independent of everything) to the balanced-fund regression. Show that $R^2$ rises, and
   compute the adjusted $R^2 = 1 - (1 - R^2)(n - 1)/(n - k)$ before and after.
5. Simulate 2,000 ten-year histories of the balanced fund and compare the standard deviation of each estimated coefficient
   with the average of the formula standard errors $s\sqrt{[(X^\top X)^{-1}]_{jj}}$.

### Advanced

6. Show that $\operatorname{Cov}(\hat b) = \sigma^2(X^\top X)^{-1}$ starting from $\hat b - b = (X^\top X)^{-1}X^\top\varepsilon$, and verify that its $(2,2)$ element equals the 07/01 formula
   $\sigma^2/\sum_t(x_t - \bar x)^2$ for one regressor plus intercept.
7. Ridge regression: $\hat b_\lambda = (X^\top X + \lambda I)^{-1}X^\top y$ (without penalizing the intercept). For the $\rho = 0.98$ two-index problem, plot the
   mean squared error of the two slopes against $\lambda$. Does a small $\lambda$ help? (Preview of shrinkage in 08/02.)

## 8. Solutions / Checks

In [ ]:
# 1.
x1, y1 = np.array([1.0, 2, 3, 4]), np.array([1.0, 3, 2, 5])
X1 = np.column_stack([np.ones(4), x1])
print("1. X'X =", (X1.T @ X1).tolist(), " X'y =", (X1.T @ y1).tolist(), " b =", np.linalg.lstsq(X1, y1, rcond=None)[0].round(4))

# 2.
print(f"2. SE multiplier {np.sqrt(1 / (1 - 0.81)):.2f}")

# 4.
def r2_adj(Xm, yv):
    ev = yv - Xm @ np.linalg.lstsq(Xm, yv, rcond=None)[0]
    r2 = 1 - ev @ ev / ((yv - yv.mean()) @ (yv - yv.mean()))
    return r2, 1 - (1 - r2) * (len(yv) - 1) / (len(yv) - Xm.shape[1])

X_noise = np.column_stack([X, rng.standard_normal(n)])
print("4. without noise regressor: R2 {:.5f}, adj {:.5f}".format(*r2_adj(X, y)))
print("   with noise regressor   : R2 {:.5f}, adj {:.5f}".format(*r2_adj(X_noise, y)))

In [ ]:
# 5.
L12 = np.linalg.cholesky(Sigma / 12)
est, ses = [], []
for p in range(2_000):
    Rp = (mu - rf) / 12 + rng.standard_normal((n, 3)) @ L12.T
    yp = alpha_fund / 12 + Rp @ w_fund + te_fund / np.sqrt(12) * rng.standard_normal(n)
    Xp = np.column_stack([np.ones(n), Rp])
    bp = np.linalg.lstsq(Xp, yp, rcond=None)[0]
    ep = yp - Xp @ bp
    est.append(bp)
    ses.append(np.sqrt(ep @ ep / (n - 4) * np.diag(np.linalg.inv(Xp.T @ Xp))))
est, ses = np.array(est), np.array(ses)
print(pd.DataFrame({"simulated SD": est.std(axis=0), "average formula SE": ses.mean(axis=0)}, index=names).round(4))

# 6.
xc = X1[:, 1] - X1[:, 1].mean()
print(f"6. [(X'X)^-1]_22 = {np.linalg.inv(X1.T @ X1)[1, 1]:.4f},  1 / sum (x - xbar)^2 = {1 / (xc @ xc):.4f}")

In [ ]:
# 7.
lams = np.logspace(-6, -1, 30)
C98 = np.array([[1, 0.98], [0.98, 1]]) * 0.16**2 / 12
L98 = np.linalg.cholesky(C98)
mse_lam = np.zeros(lams.size)
for p in range(500):
    Z = rng.standard_normal((120, 2)) @ L98.T
    yy = Z @ np.array([0.5, 0.5]) + 0.05 / np.sqrt(12) * rng.standard_normal(120)
    Zc, yc = Z - Z.mean(axis=0), yy - yy.mean()                 # centring removes the intercept
    for i, lam in enumerate(lams):
        bl = np.linalg.solve(Zc.T @ Zc + lam * np.eye(2), Zc.T @ yc)
        mse_lam[i] += np.sum((bl - 0.5) ** 2) / 500
best = lams[np.argmin(mse_lam)]
print(f"7. MSE at lambda -> 0: {mse_lam[0]:.4f}, minimum {mse_lam.min():.4f} at lambda {best:.1e}")

plt.figure(figsize=(7, 4))
plt.semilogx(lams, mse_lam)
plt.xlabel("lambda")
plt.ylabel("MSE of the two slopes")
plt.title("Ridge regression, rho = 0.98")
plt.grid(True, alpha=0.3, which="both")
plt.show()

**1.** $X^\top X = \begin{pmatrix} 4 & 10 \\ 10 & 30\end{pmatrix}$, $X^\top y = (11, 33)$, determinant 20, $\hat b = \frac{1}{20}(30\cdot 11 - 10\cdot 33,\ -10\cdot 11 + 4\cdot 33) = (0, 1.1)$.

**2.** $\sqrt{1/(1 - 0.81)} = 2.29$.

**3.** 93% of the fund's return variance is explained by a static 55/45 mix of global equities and Swiss bonds. It is
not uncorrelated with markets: it behaves like a balanced fund. Unless the 7% residual has a clearly positive mean (alpha
after fees), the investor can replicate it with two index funds at much lower cost.

**4.** $R^2$ rises from 0.97405 to 0.97427 although the new regressor is pure noise. The adjusted $R^2$ stays at 0.97338: here the
in-sample gain happens to offset the penalty exactly. A pure-noise regressor raises $R^2$ by about $(1 - R^2)/(n - k)$ on average, which is
what the adjustment charges for it, so adjusted $R^2$ does not reward it.

**5.** The simulated standard deviations and the average formula standard errors agree for all four coefficients.

**6.** $\operatorname{Cov}(\hat b) = (X^\top X)^{-1}X^\top\operatorname{Cov}(\varepsilon)X(X^\top X)^{-1} = \sigma^2(X^\top X)^{-1}$. For $X = [\mathbf 1, x]$, $(X^\top X)^{-1}_{22} = n/(n\sum x^2 - (\sum x)^2) = 1/\sum(x - \bar x)^2$.

**7.** Yes. The mean squared error of the two slopes falls from 0.043 without shrinkage to 0.0027 at $\lambda = 0.02$ (about 8% of the
diagonal of $X^\top X$). Ridge trades a little bias, pulling the two slopes toward each other and toward zero, for a large
reduction in variance along the poorly identified direction.

## 9. Conclusion

- The normal equations $X^\top X\hat b = X^\top y$ follow from setting the gradient of the squared error to zero; $\hat b = X^+y$.
- Fitted values are the orthogonal projection of $y$ onto the column space of $X$; residuals are orthogonal to every regressor.
- $\operatorname{Cov}(\hat b) = \sigma^2(X^\top X)^{-1}$ gives standard errors and t-statistics.
- Correlated regressors inflate variances by the VIF $1/(1 - R_j^2)$; their sum is well identified, the split is not.
- Solve with QR or SVD (`lstsq`), not by inverting $X^\top X$, whose condition number is the square of $X$'s.
- Style analysis is least squares with portfolio constraints.

Next: [07/03 Multi-factor models and return attribution](./03_multi_factor_models_and_return_attribution.ipynb).